# 2.1 Build Australian Centre for Advanced Photovoltaics (ACAP), Energy Data Platform (ACAP-EDP) Per-Household 30-Minute Checkpoints

## Purpose
Build full-period per-household ACAP-EDP checkpoints for the aggregation-level experiment. This notebook uses only top-level raw ACAP-EDP CSV files whose filenames encode months from `2021-07` through `2024-06`.

It intentionally does not scan nested folders such as `ACAP EDP/2021 Jul to 2024 Jun`, and it does not open irrelevant months such as `2019-07`.

## Inputs
- `SITE_LIST_PATH`

## Run Flow
1. Purpose, Inputs, And Outputs
2. Recover The ACAP-EDP Cohort
3. Select Only Required Top-Level Monthly Raw Files
4. Scan Selected Months Into An In-Memory Native Grid
5. Build Full-Period Per-Household 30-Minute Checkpoints
6. Validate Final Checkpoints
7. Completion Notes

## Outputs
- `AUDIT_DIR / "acap_edp_148hh_recovered_site_list.csv"`
- `DATA_EXPLORATION_DIR / "acap_edp_148hh_recovered_site_list.csv"`
- `AUDIT_DIR / "acap_edp_checkpoint_selected_top_level_raw_files.csv"`
- `DATA_EXPLORATION_DIR / "acap_edp_checkpoint_selected_top_level_raw_files.csv"`
- `AUDIT_DIR / "acap_edp_checkpoint_ignored_top_level_raw_files.csv"`
- `AUDIT_DIR / "acap_edp_checkpoint_monthly_scan_summary.csv"`
- `DATA_EXPLORATION_DIR / "acap_edp_checkpoint_monthly_scan_summary.csv"`
- `AUDIT_DIR / "acap_edp_site_30min_checkpoint_summary.csv"`
- `AUDIT_DIR / "acap_edp_valid_checkpoint_sites.csv"`
- `AUDIT_DIR / "acap_edp_excluded_checkpoint_sites.csv"`

## 1. Purpose, Inputs, And Outputs

Inputs:

- recovered ACAP-EDP 148-household cohort spreadsheet,
- top-level monthly ACAP-EDP raw CSVs in the required period,
- `ac_load_net` circuit rows only.

Outputs:

- one compressed 30-minute Parquet checkpoint per valid household,
- audit tables for selected files, monthly scans, valid checkpoints, and excluded sites.

The checkpoint fill order is household-level: each household is converted to a native 5-minute power series, forward-filled across the full period, and then resampled to 30 minutes. Notebook `2.2` will aggregate these already-complete household series.


In [ ]:
import os
from pathlib import Path
import re
import sys

import numpy as np
import pandas as pd


def find_publication_project(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "specs").exists() and (candidate / "data").exists() and (candidate / "results").exists():
            return candidate
    raise FileNotFoundError("Could not find publication/journal_article_1 from the current working directory.")


def find_repo_root(start: Path) -> Path:
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "pynnlf").exists():
            return candidate
    raise FileNotFoundError("Could not find the PyNNLF repo root.")


PROJECT_DIR = find_publication_project()
REPO_ROOT = find_repo_root(PROJECT_DIR)
print(f"Publication project: {PROJECT_DIR}")
print(f"Repository root: {REPO_ROOT}")

def raw_data_root():
    """Locate the local source data tree, which is not distributed with this repository.

    Set the PYNNLF_RAW_DATA_DIR environment variable to the directory holding the
    "1. raw", "2. processed" and "3. cleaned" folders before running this notebook.

    Returns:
        Path: root of the local source data tree.
    """
    root = os.environ.get("PYNNLF_RAW_DATA_DIR")
    if not root:
        raise RuntimeError(
            "PYNNLF_RAW_DATA_DIR is not set. Point it at your local source data "
            "directory; see the Data section of the repository README."
        )
    return Path(root)


RAW_DATA_ROOT = raw_data_root()

RAW_ACAP_EDP_DIR = RAW_DATA_ROOT / "1. raw" / "ACAP EDP"
PROCESSED_DIR = RAW_DATA_ROOT / "2. processed"
CLEANED_WORKSPACE = RAW_DATA_ROOT / "3. cleaned" / "ACAP_EDP_different_aggregation"
SITE_LIST_PATH = PROCESSED_DIR / "acap_edp_cluster_2_2_3years.xlsx"

CHECKPOINT_DIR = CLEANED_WORKSPACE / "checkpoints" / "site_30min"
AUDIT_DIR = CLEANED_WORKSPACE / "audits"
RESULTS_DIR = PROJECT_DIR / "results" / "03_acap_edp_aggregation_level"
DATA_EXPLORATION_DIR = RESULTS_DIR / "01_data_exploration"
for directory in [CHECKPOINT_DIR, AUDIT_DIR, RESULTS_DIR, DATA_EXPLORATION_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

START = pd.Timestamp("2021-07-01 00:00:00")
END_30MIN = pd.Timestamp("2024-06-30 23:30:00")
END_NATIVE = END_30MIN + pd.Timedelta(minutes=25)
INDEX_NATIVE = pd.date_range(START, END_NATIVE, freq="5min", name="datetime")
INDEX_30MIN = pd.date_range(START, END_30MIN, freq="30min", name="datetime")
REQUIRED_MONTHS = list(pd.period_range("2021-07", "2024-06", freq="M"))
RAW_INTERVAL_SECONDS = 300
RAW_INTERVAL_HOURS = pd.Timedelta(minutes=5) / pd.Timedelta(hours=1)
CHUNKSIZE = 2_000_000
MIN_VALID_SITES = 100

for path in [RAW_ACAP_EDP_DIR, SITE_LIST_PATH]:
    if not path.exists():
        raise FileNotFoundError(path)

print(f"Raw ACAP-EDP directory: {RAW_ACAP_EDP_DIR}")
print(f"Checkpoint directory: {CHECKPOINT_DIR}")
print(f"Study period: {START} to {END_30MIN}")
print(f"Native 5-minute rows: {len(INDEX_NATIVE):,}")
print(f"30-minute rows per checkpoint: {len(INDEX_30MIN):,}")


## 2. Recover The ACAP-EDP Cohort

The site list is the recovered 148-household ACAP-EDP cohort used by the existing ACAP-EDP aggregate dataset. The checkpoint builder keeps the full cohort for audit purposes, then marks sites valid only if they produce complete `ac_load_net` checkpoints.


In [ ]:
site_list = pd.read_excel(SITE_LIST_PATH)
required_columns = {"edp_site_id", "postcode", "state", "lat_deg", "long_deg", "date_of_first_data", "date_of_last_data", "number_of_days"}
missing_columns = required_columns - set(site_list.columns)
if missing_columns:
    raise ValueError(f"ACAP-EDP site list is missing required columns: {sorted(missing_columns)}")

site_list["edp_site_id"] = site_list["edp_site_id"].astype(str).str.strip()
site_ids = site_list["edp_site_id"].tolist()
if len(site_ids) != 148 or len(set(site_ids)) != 148:
    raise ValueError(f"Expected 148 unique ACAP-EDP site IDs, found {len(site_ids)} rows and {len(set(site_ids))} unique IDs")

site_list.to_csv(AUDIT_DIR / "acap_edp_148hh_recovered_site_list.csv", index=False)
site_list.to_csv(DATA_EXPLORATION_DIR / "acap_edp_148hh_recovered_site_list.csv", index=False)
print(f"Recovered ACAP-EDP cohort sites: {len(site_ids)}")
display(site_list.head())


## 3. Select Only Required Top-Level Monthly Raw Files

This is the disk-safety gate. The notebook enumerates top-level filenames only with `RAW_ACAP_EDP_DIR.glob("edp_data_*.csv")`, parses the encoded month from the filename, and only then selects the 36 required months. It does not use `rglob`, so nested duplicate folders are ignored.


In [ ]:
MONTH_RE = re.compile(r"^edp_data_(\d{4})_(\d{2})")


def encoded_year_month(path: Path) -> pd.Period | None:
    match = MONTH_RE.match(path.stem)
    if not match:
        return None
    year = int(match.group(1))
    month = int(match.group(2))
    if not 1 <= month <= 12:
        raise ValueError(f"Invalid encoded month in filename: {path.name}")
    return pd.Period(year=year, month=month, freq="M")


# Top-level only. Do not replace this with rglob; nested OneDrive duplicates must remain untouched.
top_level_candidates = sorted(RAW_ACAP_EDP_DIR.glob("edp_data_*.csv"))
records = []
ignored_records = []
selected_by_month = {}
required_set = set(REQUIRED_MONTHS)

for path in top_level_candidates:
    if path.parent != RAW_ACAP_EDP_DIR:
        raise RuntimeError(f"Unexpected nested path reached top-level scan: {path}")
    encoded_month = encoded_year_month(path)
    record = {
        "filename": path.name,
        "raw_file": str(path),
        "encoded_month": None if encoded_month is None else str(encoded_month),
        "selected": encoded_month in required_set,
    }
    if encoded_month in required_set:
        if encoded_month in selected_by_month:
            raise ValueError(f"Duplicate top-level files for {encoded_month}: {selected_by_month[encoded_month].name} and {path.name}")
        selected_by_month[encoded_month] = path
        records.append(record)
    else:
        ignored_records.append(record)

missing_months = [str(month) for month in REQUIRED_MONTHS if month not in selected_by_month]
if missing_months:
    raise FileNotFoundError(f"Missing required top-level ACAP-EDP raw files for months: {missing_months}")

selected_raw_files = [selected_by_month[month] for month in REQUIRED_MONTHS]
selected_file_audit = pd.DataFrame([
    {
        "encoded_month": str(month),
        "filename": selected_by_month[month].name,
        "raw_file": str(selected_by_month[month]),
        "parent_directory": str(selected_by_month[month].parent),
        "size_gb": selected_by_month[month].stat().st_size / 1e9,
    }
    for month in REQUIRED_MONTHS
])
ignored_file_audit = pd.DataFrame(ignored_records)

if selected_file_audit["raw_file"].str.contains(r"2021 Jul to 2024 Jun", regex=False).any():
    raise RuntimeError("Nested duplicate raw folder appeared in the selected-file audit. This notebook must use top-level files only.")

selected_file_audit.to_csv(AUDIT_DIR / "acap_edp_checkpoint_selected_top_level_raw_files.csv", index=False)
selected_file_audit.to_csv(DATA_EXPLORATION_DIR / "acap_edp_checkpoint_selected_top_level_raw_files.csv", index=False)
ignored_file_audit.to_csv(AUDIT_DIR / "acap_edp_checkpoint_ignored_top_level_raw_files.csv", index=False)

print(f"Top-level raw candidates seen by filename: {len(top_level_candidates)}")
print(f"Selected required monthly files: {len(selected_raw_files)}")
for path in selected_raw_files:
    print(f"  {path.name}")
display(selected_file_audit)


## 4. Scan Selected Months Into An In-Memory Native Grid

The selected raw CSVs are processed one month at a time. Only required columns are read, and only `ac_load_net` rows for the recovered cohort are kept. The result is held in memory as a site-by-time 5-minute energy grid so no 5-minute checkpoint files are written to disk.


In [ ]:
site_index = {site_id: idx for idx, site_id in enumerate(site_ids)}
start_unix = int(START.tz_localize("UTC").timestamp())
end_unix = int(END_NATIVE.tz_localize("UTC").timestamp())
n_sites = len(site_ids)
n_native = len(INDEX_NATIVE)

energy_sum = np.zeros((n_sites, n_native), dtype=np.float64)
present = np.zeros((n_sites, n_native), dtype=bool)
usecols = ["edp_site_id", "unix_time", "circuit_label", "real_energy"]
site_id_set = set(site_ids)
scan_rows = []

for raw_file in selected_raw_files:
    chunk_count = 0
    raw_rows = 0
    filtered_rows = 0
    grouped_rows = 0
    invalid_timestamp_rows = 0
    print(f"[start] {raw_file.name}")
    try:
        for chunk in pd.read_csv(raw_file, usecols=usecols, chunksize=CHUNKSIZE):
            chunk_count += 1
            raw_rows += chunk.shape[0]
            chunk["edp_site_id"] = chunk["edp_site_id"].astype(str).str.strip()
            chunk = chunk.loc[
                chunk["edp_site_id"].isin(site_id_set)
                & chunk["circuit_label"].eq("ac_load_net")
                & chunk["unix_time"].between(start_unix, end_unix)
            ]
            if chunk.empty:
                continue
            filtered_rows += chunk.shape[0]
            unix_values = chunk["unix_time"].to_numpy(dtype=np.int64)
            valid_grid = ((unix_values - start_unix) % RAW_INTERVAL_SECONDS) == 0
            invalid_timestamp_rows += int((~valid_grid).sum())
            if not valid_grid.all():
                chunk = chunk.loc[valid_grid]
                if chunk.empty:
                    continue
            grouped = chunk.groupby(["edp_site_id", "unix_time"], observed=True, as_index=False)["real_energy"].sum()
            grouped_rows += grouped.shape[0]
            site_positions = grouped["edp_site_id"].map(site_index).to_numpy(dtype=np.int64)
            time_positions = ((grouped["unix_time"].to_numpy(dtype=np.int64) - start_unix) // RAW_INTERVAL_SECONDS).astype(np.int64)
            values = grouped["real_energy"].to_numpy(dtype=np.float64)
            if (time_positions < 0).any() or (time_positions >= n_native).any():
                raise ValueError(f"{raw_file.name}: timestamp position outside expected native index")
            np.add.at(energy_sum, (site_positions, time_positions), values)
            present[site_positions, time_positions] = True
    except OSError as exc:
        raise OSError(f"Could not read selected in-scope raw file {raw_file.name}. If this is a OneDrive hydration issue, free space or hydrate only this file, then rerun.") from exc

    scan_rows.append({
        "filename": raw_file.name,
        "chunks": chunk_count,
        "raw_rows_seen": raw_rows,
        "filtered_ac_load_net_rows": filtered_rows,
        "grouped_site_timestamp_rows": grouped_rows,
        "invalid_timestamp_rows_dropped": invalid_timestamp_rows,
    })
    print(f"[done] {raw_file.name}: chunks={chunk_count}, filtered_rows={filtered_rows:,}, grouped_rows={grouped_rows:,}")

monthly_scan_summary = pd.DataFrame(scan_rows)
monthly_scan_summary.to_csv(AUDIT_DIR / "acap_edp_checkpoint_monthly_scan_summary.csv", index=False)
monthly_scan_summary.to_csv(DATA_EXPLORATION_DIR / "acap_edp_checkpoint_monthly_scan_summary.csv", index=False)
if monthly_scan_summary["grouped_site_timestamp_rows"].sum() == 0:
    raise ValueError("No ac_load_net rows were extracted from the selected months.")

display(monthly_scan_summary)


## 5. Build Full-Period Per-Household 30-Minute Checkpoints

Each household is converted from 5-minute energy to 5-minute power, forward-filled across the full study period, and resampled to 30-minute mean power. Sites with no usable `ac_load_net` data or unresolved leading gaps are excluded and audited.


In [ ]:
checkpoint_rows = []
valid_site_ids = []

for site_id, idx in site_index.items():
    site_present = present[idx]
    raw_points = int(site_present.sum())
    row = {
        "edp_site_id": site_id,
        "raw_5min_points": raw_points,
        "status": "pending",
        "first_raw_datetime": pd.NaT,
        "last_raw_datetime": pd.NaT,
        "native_missing_before_ffill": int((~site_present).sum()),
        "native_missing_after_ffill": pd.NA,
        "checkpoint_rows": 0,
        "checkpoint_path": "",
    }
    if raw_points == 0:
        row["status"] = "excluded_no_ac_load_net_rows"
        checkpoint_rows.append(row)
        continue

    first_pos = int(np.flatnonzero(site_present)[0])
    last_pos = int(np.flatnonzero(site_present)[-1])
    row["first_raw_datetime"] = INDEX_NATIVE[first_pos]
    row["last_raw_datetime"] = INDEX_NATIVE[last_pos]

    native_energy = pd.Series(energy_sum[idx].copy(), index=INDEX_NATIVE, name="netload_kWh_5min_equivalent")
    native_energy.loc[~site_present] = np.nan
    native_power = native_energy.div(1000.0).div(RAW_INTERVAL_HOURS).rename("netload_kW")
    native_power = native_power.ffill()
    missing_after_ffill = int(native_power.isna().sum())
    row["native_missing_after_ffill"] = missing_after_ffill
    if missing_after_ffill > 0:
        row["status"] = "excluded_leading_missing_after_ffill"
        checkpoint_rows.append(row)
        continue

    site_30min = native_power.resample("30min", label="left", closed="left").mean().reindex(INDEX_30MIN).reset_index()
    site_30min.insert(0, "edp_site_id", site_id)
    if site_30min.shape[0] != len(INDEX_30MIN):
        row["status"] = "excluded_wrong_row_count"
        checkpoint_rows.append(row)
        continue
    if site_30min["datetime"].duplicated().any() or site_30min["netload_kW"].isna().any():
        row["status"] = "excluded_invalid_checkpoint_values"
        checkpoint_rows.append(row)
        continue

    out_path = CHECKPOINT_DIR / f"{site_id}_30min.parquet"
    site_30min.to_parquet(out_path, index=False, compression="snappy")
    row["status"] = "valid"
    row["checkpoint_rows"] = site_30min.shape[0]
    row["checkpoint_path"] = str(out_path)
    valid_site_ids.append(site_id)
    checkpoint_rows.append(row)

checkpoint_summary = pd.DataFrame(checkpoint_rows).merge(
    site_list[["edp_site_id", "postcode", "state", "lat_deg", "long_deg", "date_of_first_data", "date_of_last_data", "number_of_days"]],
    on="edp_site_id",
    how="left",
)
valid_checkpoint_sites = checkpoint_summary.loc[checkpoint_summary["status"].eq("valid")].copy()
excluded_sites = checkpoint_summary.loc[~checkpoint_summary["status"].eq("valid")].copy()

checkpoint_summary.to_csv(AUDIT_DIR / "acap_edp_site_30min_checkpoint_summary.csv", index=False)
valid_checkpoint_sites.to_csv(AUDIT_DIR / "acap_edp_valid_checkpoint_sites.csv", index=False)
excluded_sites.to_csv(AUDIT_DIR / "acap_edp_excluded_checkpoint_sites.csv", index=False)
checkpoint_summary.to_csv(DATA_EXPLORATION_DIR / "acap_edp_site_30min_checkpoint_summary.csv", index=False)
valid_checkpoint_sites.to_csv(DATA_EXPLORATION_DIR / "acap_edp_valid_checkpoint_sites.csv", index=False)
excluded_sites.to_csv(DATA_EXPLORATION_DIR / "acap_edp_excluded_checkpoint_sites.csv", index=False)

print(f"Valid checkpoint sites: {len(valid_site_ids)}")
print(f"Excluded sites: {excluded_sites.shape[0]}")
if len(valid_site_ids) < MIN_VALID_SITES:
    raise ValueError(f"Only {len(valid_site_ids)} valid checkpoint sites were produced; at least {MIN_VALID_SITES} are required for 100hh sampling.")

display(checkpoint_summary["status"].value_counts().rename_axis("status").reset_index(name="site_count"))
display(valid_checkpoint_sites.head())
if not excluded_sites.empty:
    display(excluded_sites[["edp_site_id", "status", "raw_5min_points", "first_raw_datetime", "last_raw_datetime"]])


## 6. Validate Final Checkpoints

This cell reloads the written Parquet files and confirms the final shape expected by notebook `2.2`.


In [ ]:
validation_rows = []
for site_id in valid_site_ids:
    path = CHECKPOINT_DIR / f"{site_id}_30min.parquet"
    frame = pd.read_parquet(path)
    spacing_ok = frame["datetime"].diff().dropna().eq(pd.Timedelta(minutes=30)).all()
    validation_rows.append({
        "edp_site_id": site_id,
        "rows": frame.shape[0],
        "start": frame["datetime"].min(),
        "end": frame["datetime"].max(),
        "spacing_ok": bool(spacing_ok),
        "duplicate_timestamps": int(frame["datetime"].duplicated().sum()),
        "missing_netload": int(frame["netload_kW"].isna().sum()),
    })

checkpoint_validation = pd.DataFrame(validation_rows)
checkpoint_validation.to_csv(AUDIT_DIR / "acap_edp_site_30min_checkpoint_validation.csv", index=False)
checkpoint_validation.to_csv(DATA_EXPLORATION_DIR / "acap_edp_site_30min_checkpoint_validation.csv", index=False)

bad = checkpoint_validation.loc[
    checkpoint_validation["rows"].ne(len(INDEX_30MIN))
    | ~checkpoint_validation["spacing_ok"]
    | checkpoint_validation["duplicate_timestamps"].ne(0)
    | checkpoint_validation["missing_netload"].ne(0)
]
if not bad.empty:
    raise ValueError("Some checkpoint files failed validation", bad.head())

display(checkpoint_validation.head())
print("All valid checkpoint files passed validation.")


## 7. Completion Notes

Notebook `2.1` is complete when the valid checkpoint site list and per-household Parquet files have been written. Notebook `2.2` should sample from `acap_edp_valid_checkpoint_sites.csv`, not directly from all 148 recovered sites.
